In [1]:
# %%
"""
نوتبوک: 06_test_main_graph_with_tools.ipynb (FINAL VERSION)
هدف: تست end-to-end گراف مولتی‌ایجنت با parser اصلاح شده
"""

import os
import sys
from pathlib import Path

# تنظیم مسیر پروژه
project_root = Path.cwd().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))

print(f"✓ Project root: {project_root}")
print(f"✓ Src path: {src_path}")

from dotenv import load_dotenv
load_dotenv()

print("✓ OPENROUTER_API_KEY:", (os.getenv("OPENROUTER_API_KEY") or "")[:20], "...")
print("✓ COHERE_API_KEY:", (os.getenv("COHERE_API_KEY") or "")[:20], "...")

# %%
"""
🔄 FORCE RELOAD (اجباری برای اطمینان از کد جدید)
"""
import importlib

modules_to_reload = [
    'legal_multi_agent.utils.toon',  # ⭐ اول این را reload کن
    'legal_multi_agent.state.schemas',
    'legal_multi_agent.agents.supervisor',
    'legal_multi_agent.agents.researcher',
    'legal_multi_agent.agents.reasoner',
    'legal_multi_agent.agents.critic',
    'legal_multi_agent.agents.tool_executor',
    'legal_multi_agent.graphs.main_graph',
]

print("🔄 Reloading modules...")
for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"   ♻️  Reloaded: {module_name}")

print("✅ All modules reloaded\n")

# %%
"""
🧪 TEST PARSER (اطمینان از fallback)
"""
from legal_multi_agent.utils.toon import extract_toon_answer

test_llm_output = "results{توضیحات تست با کاما، نقطه‌ویرگول؛ و خط فاصله - همه باید کار کند,3,4}"

result = extract_toon_answer(test_llm_output, verbose=True)
print(f"\n✅ Parser test result: {result}")

if result and result['answer'] == '3':
    print("✅ Parser is working correctly!")
else:
    print("❌ Parser still has issues")

# %%
"""
📦 IMPORT GRAPH
"""
from legal_multi_agent.graphs.main_graph import graph

print("✅ Graph imported")

# %%
"""
🧪 SIMPLE TEST (بدون tools)
"""
print("="*70)
print("🧪 TEST 1: Simple test without tools")
print("="*70)

simple_state = {
    "question": "تست ساده",
    "options_text": "1) یک\n2) دو\n3) سه\n4) چهار",
    "max_revisions": 1,
    "use_option_verifier": False,
    "use_retriever_tool": False,
}

print("\n⏳ Running simple test...")
step = 0
final_state = None

for chunk in graph.stream(simple_state, {"recursion_limit": 15}):
    step += 1
    node_name = list(chunk.keys())[0]
    node_state = list(chunk.values())[0]
    
    print(f"   Step {step}: {node_name}")
    
    # چک کردن draft_toon
    if node_state.get("draft_toon"):
        print(f"      ✅ draft_toon: {node_state['draft_toon']}")
    
    final_state = node_state
    
    if step >= 12:
        print("   ⚠️  Stopping at step 12 for test")
        break

if final_state and final_state.get("final_toon"):
    print(f"\n✅ SUCCESS! Final answer: {final_state['final_toon']['answer']}")
else:
    print(f"\n⚠️  No final answer yet")

print("\n" + "="*70 + "\n")

# %%
"""
🚀 FULL TEST (با tools)
"""

def run_full_test(
    question_number: int,
    question: str,
    options_text: str,
    max_revisions: int = 2,
):
    """
    تست کامل با دیباگ
    """
    initial_state = {
        "question_number": question_number,
        "question": question,
        "options_text": options_text,
        "max_revisions": max_revisions,
        "use_option_verifier": True,
        "use_retriever_tool": True,
    }
    
    print("="*70)
    print(f"🎯 سوال شماره {question_number}")
    print("="*70)
    print(question[:100] + "...")
    print("\n⏳ شروع اجرا...")
    print("="*70)
    
    step_count = 0
    final_state = None
    
    for state_snapshot in graph.stream(
        initial_state,
        {"recursion_limit": 30},
        stream_mode="values"
    ):
        step_count += 1
        
        print(f"\n📍 Step {step_count}:")
        print(f"   Context: {len(state_snapshot.get('context', ''))} chars")
        print(f"   Draft: {bool(state_snapshot.get('draft_toon'))}")
        print(f"   Final: {bool(state_snapshot.get('final_toon'))}")
        print(f"   Next: {state_snapshot.get('next')}")
        
        # نمایش draft اگر تازه تولید شده
        if state_snapshot.get('draft_toon') and not (final_state and final_state.get('draft_toon')):
            draft = state_snapshot['draft_toon']
            print(f"   🎯 DRAFT GENERATED: answer={draft['answer']}, conf={draft['confidence']}")
        
        final_state = state_snapshot
        
        # موفقیت
        if state_snapshot.get('final_toon'):
            print(f"\n✅ SUCCESS!")
            break
        
        if step_count >= 25:
            print("\n❌ به recursion limit رسیدیم!")
            break
    
    print("\n" + "="*70)
    print(f"✅ اجرا در {step_count} step تمام شد")
    print("="*70)
    
    # نمایش نتیجه
    if final_state and final_state.get("final_toon"):
        print("\n🎯 پاسخ نهایی:")
        final = final_state["final_toon"]
        print(f"   جواب: گزینه {final.get('answer')}")
        print(f"   اطمینان: {final.get('confidence')}/5")
        print(f"   توضیح: {final.get('explanation', '')[:200]}...")
    else:
        print("\n⚠️  پاسخ نهایی تولید نشد")
    
    return final_state


# اجرا
q1 = """شخص «الف» حین رانندگی با خودرو سواری با شخص «ب» (موتورسوار) تصادف می کند شخص «الف» که دارای پوشش بیمه اجباری است صد درصد مقصر حادثه تشخیص داده می شود. بیمهگر مسئولیت، تکلیف به جبران کدام دسته از خسارات و تا چه میزانی دارد؟"""

opts1 = """
1) دیه هر دو راننده صرفاً تا میزان دیه کامل و خسارات مالی وارد شده به موتورسیکلت تا سقف تعهدات 
2) دیه شخص «الف» تا سقف تعهدات خود و دیه شخص «ب» به هر میزان که طبق قانون تعلق گیرد و خسارات مالی وارده به موتورسیکلت تا سقف تعهدات خود 
3) دیه هر دو راننده تا هر میزان که باشد و خسارت وارد شده به هر دو وسیله نقلیه به صورت کامل 
4) دیه هر دو راننده تا هر میزان که باشد و خسارت وارد شده به هر دو وسیله نقلیه تا سقف تعهدات
"""

result = run_full_test(
    question_number=2,
    question=q1,
    options_text=opts1,
)

# %%


✓ Project root: f:\Thesis\project\3-Multi-Agent-System
✓ Src path: f:\Thesis\project\3-Multi-Agent-System\src
✓ OPENROUTER_API_KEY: sk-or-v1-10d2ef641ff ...
✓ COHERE_API_KEY: O0pIM8Zkm973sO7HkLSO ...
🔄 Reloading modules...
✅ All modules reloaded

⚠️ TOON: header not found for schema=['explanation', 'answer', 'confidence']
⚠️ TOON(answer): standard format failed, trying fallback...
✅ TOON(answer): fallback successful - answer=3, conf=4

✅ Parser test result: {'explanation': 'توضیحات تست با کاما، نقطه\u200cویرگول؛ و خط فاصله - همه باید کار کند', 'answer': '3', 'confidence': 4}
✅ Parser is working correctly!
✅ Graph imported
🧪 TEST 1: Simple test without tools

⏳ Running simple test...

🔷 ═══ INITIALIZE START ═══
   ✓ Set revision_count = 0
   ✓ Initialized tool_results = {}
🔷 ═══ INITIALIZE END ═══

   Step 1: initialize

🟢 ═══ SUPERVISOR START ═══
   📊 State overview:
      - has_context: False
      - has_draft_toon: False
      - has_critic_toon: False
      - has_final_toon: False
  